# Regression - Debugging Workshop

Days 6-12 built the full regression toolkit: a baseline hierarchy (Dummy → OLS → Ridge → KNN), cross-validation, Pipelines to prevent leakage, and `validation_curve`/`GridSearchCV` for tuning. This lecture is synthesis. We apply the same toolkit to four datasets with planted problems and practice reading the failure signals that appear when models break - the pattern recognition that transfers directly to unscaffolded work on the project.

## Setup

In [ ]:
%matplotlib inline

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from sklearn.datasets import make_friedman1
from sklearn.dummy import DummyRegressor
from sklearn.linear_model import LinearRegression, RidgeCV, LassoCV
from sklearn.model_selection import (
    cross_val_score, KFold, GridSearchCV,
    learning_curve, validation_curve,
    train_test_split,
)
from sklearn.neighbors import KNeighborsRegressor
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

All four scenarios use the same CV strategy. Boston serves as a sanity reference when introducing `learning_curve`.

In [ ]:
url = "https://raw.githubusercontent.com/olearydj/INSY7120/refs/heads/main/notebooks/data/Boston.csv"
boston = pd.read_csv(url)
X_boston = boston.loc[:, 'zn':]
y_boston = boston['crim']

cv = KFold(n_splits=5, shuffle=True, random_state=42)

## Diagnostic Signals

Before seeing specific failures, we need a vocabulary for what "wrong" looks like. The table below is a field reference. Each signal will appear concretely in the scenarios that follow.

| Signal | Likely cause | Next step |
|--------|-------------|-----------|
| Negative R² | Model worse than DummyRegressor | Check preprocessing; run the full hierarchy to isolate the problem |
| Compressed KNN predictions | Scaling failure or severe leakage | Verify Pipeline includes StandardScaler for KNN and regularized models |
| Train score >> Test score | Overfitting | Reduce complexity; use validation_curve or regularization |
| High CV fold variance | Small dataset or unstable model | Check n; try a simpler model; increase cv folds |
| Adding features hurts | Noise or curse of dimensionality | Compare KNN vs linear performance as feature count grows |
| Unstable OLS coefficients | Collinearity | Inspect correlation matrix; compare OLS vs Ridge coefficient distributions |

## `learning_curve`

This lecture introduces `learning_curve`, the third member of the diagnostic trio:

| Tool | What varies | What it answers |
|------|-------------|-----------------|
| `validation_curve` | Hyperparameter value | Is the model too simple or too complex? |
| `GridSearchCV` | Hyperparameter(s) | What setting performs best? |
| `learning_curve` | Training set size | Is performance limited by data or by model class? |

The key question `learning_curve` answers: would more data help? If the train-test gap narrows as $n$ grows, more observations will improve the model. If the gap persists at large $n$, the bottleneck is model class, not data quantity.

### The API

The signature mirrors `validation_curve`. Instead of sweeping a parameter range, it sweeps training set fractions and returns scores at each size.

In [ ]:
train_sizes, train_scores, test_scores = learning_curve(
    make_pipeline(StandardScaler(), LinearRegression()),
    X_boston, y_boston,
    train_sizes=np.linspace(0.1, 1.0, 10),
    cv=cv,
    scoring='r2',
)

# train_sizes: actual sample counts used at each step
# train_scores: shape (n_sizes, n_folds) - score on training portion of each fold
# test_scores:  shape (n_sizes, n_folds) - score on held-out portion of each fold
print(f"Training sizes: {train_sizes}")
print(f"Score array shape: {train_scores.shape}")

A helper function plots the averaged curves with a confidence band. We will call it throughout the workshop.

In [ ]:
def plot_learning_curve(estimator, X, y, title, cv, ax=None):
    if ax is None:
        _, ax = plt.subplots(figsize=(7, 4))

    sizes, tr_scores, te_scores = learning_curve(
        estimator, X, y,
        train_sizes=np.linspace(0.1, 1.0, 10),
        cv=cv, scoring='r2',
    )

    tr_mean, tr_std = tr_scores.mean(axis=1), tr_scores.std(axis=1)
    te_mean, te_std = te_scores.mean(axis=1), te_scores.std(axis=1)

    ax.plot(sizes, tr_mean, 'o-', label='Training R²')
    ax.fill_between(sizes, tr_mean - tr_std, tr_mean + tr_std, alpha=0.1)
    ax.plot(sizes, te_mean, 's--', label='CV R²')
    ax.fill_between(sizes, te_mean - te_std, te_mean + te_std, alpha=0.1)

    ax.set_xlabel('Training set size')
    ax.set_ylabel('R²')
    ax.set_title(title)
    ax.legend(loc='lower right')
    ax.grid(True, alpha=0.3)
    return ax

### Three patterns

Every learning curve takes one of three characteristic shapes. The figure below shows each, using synthetic data with known properties so the cause of each shape is unambiguous.

In [ ]:
rng_demo = np.random.default_rng(0)
n_demo = 600

# 1D quadratic signal: y = x² + noise
X_p = rng_demo.uniform(-2, 2, (n_demo, 1))
y_p = X_p.ravel()**2 + rng_demo.normal(0, 0.5, n_demo)

# same y, but X padded with 30 noise features to break distance-based methods
X_p_noisy = np.hstack([X_p, rng_demo.normal(0, 1, (n_demo, 30))])

fig, axes = plt.subplots(1, 3, figsize=(15, 4))

plot_learning_curve(
    make_pipeline(StandardScaler(), LinearRegression()),
    X_p, y_p,
    'High Bias (underfitting)\nOLS on quadratic data',
    cv, ax=axes[0],
)
plot_learning_curve(
    make_pipeline(StandardScaler(), KNeighborsRegressor(n_neighbors=1)),
    X_p_noisy, y_p,
    'High Variance (overfitting)\nKNN k=1, 31 features (30 noise)',
    cv, ax=axes[1],
)
plot_learning_curve(
    make_pipeline(StandardScaler(), KNeighborsRegressor(n_neighbors=15)),
    X_p, y_p,
    'Data-Limited\nKNN k=15 on quadratic data',
    cv, ax=axes[2],
)

plt.tight_layout()
plt.show()

What do these three learning curves tell you?

***

##### Answer

*TLDR: Three distinct shapes map to three distinct problems - model too simple, model too complex, not enough data.*

- *High bias (left)*: training and CV scores converge at the same low value and stay there. The model is too simple to capture the quadratic signal. Adding more data will not help because the model class cannot represent the true function. Fix: use a more complex model or engineer better features (polynomial terms in this case).
- *High variance (center)*: training R² is exactly 1.0 at all training sizes while CV R² is near zero, with the gap persisting at large $n$. KNN k=1 memorizes each training point (self-nearest-neighbor), but 30 noise features make distances meaningless so test neighbors are essentially random. Fix: remove irrelevant features, reduce model complexity, or add regularization.
- *Data-limited (right)*: a gap exists between training and CV scores, but it narrows as $n$ grows. The model class is appropriate but neighborhoods are sparse at small $n$. Fix: collect more data - performance will continue to improve.

***

### Boston baseline

For reference: Ridge (CV) and KNN k=1 on the Boston dataset illustrate the contrast between a well-calibrated model and an overfit one.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

plot_learning_curve(
    make_pipeline(StandardScaler(), RidgeCV(alphas=np.logspace(-3, 3, 50))),
    X_boston, y_boston,
    'Boston - Ridge (CV)',
    cv, ax=axes[0],
)
plot_learning_curve(
    make_pipeline(StandardScaler(), KNeighborsRegressor(n_neighbors=1)),
    X_boston, y_boston,
    'Boston - KNN k=1',
    cv, ax=axes[1],
)

plt.tight_layout()
plt.show()

What pattern does each curve show, and what action does each call for?

***

##### Answer

*TLDR: Ridge is data-limited (more data would help); KNN k=1 is overfit (reduce complexity first).*

Ridge shows the data-limited pattern: a gap that narrows as $n$ grows, with both scores at moderate R². Performance is constrained by data quantity, not model class. Action: with more observations, Ridge CV R² would continue to improve. No model change needed.

KNN k=1 shows high variance: training R² is exactly 1.0 at every size (each training point is its own nearest neighbor at distance zero) while CV R² is substantially lower and barely improves with more data. Action: the persistent gap signals overfitting. Use `validation_curve` or `GridSearchCV` to find a better k - the problem is model complexity, not data quantity.

***

## Debugging Scenarios

Each scenario presents a black-box dataset. The workflow for each:

1. Run the baseline hierarchy
2. Identify the signal in the output
3. Form a diagnosis
4. Confirm (the planted problem is revealed)
5. Apply a fix and verify improvement

## Scenario A - Scaling Failure

A dataset with four features: household income (dollars), age, years of education, and family size. The target is a wellbeing score. Run the hierarchy - but notice that KNN is used without a Pipeline.

Generate synthetic data so underlying distributions and modeling problems are known:

In [ ]:
rng_a = np.random.default_rng(1)
n_a = 300

income    = rng_a.uniform(20_000, 150_000, n_a)   # range ~130,000
age       = rng_a.uniform(18, 80, n_a)              # range ~62
education = rng_a.uniform(8, 22, n_a)               # range ~14
family    = rng_a.integers(1, 8, n_a).astype(float) # range ~7

X_a = pd.DataFrame({
    'income': income, 'age': age,
    'education': education, 'family_size': family,
})

# ground truth: wellbeing depends on age and education, not income
y_a = (age - 49) / 10 + (education - 15) / 3 + rng_a.normal(0, 0.4, n_a)

In [ ]:
models_a = {
    'Dummy':      make_pipeline(DummyRegressor(strategy='mean')),
    'OLS':        make_pipeline(StandardScaler(), LinearRegression()),
    'Ridge (CV)': make_pipeline(StandardScaler(), RidgeCV(alphas=np.logspace(-3, 3, 50))),
    'KNN (raw)':  make_pipeline(KNeighborsRegressor(n_neighbors=5)),  # no StandardScaler
}

print(f"{'Model':<20} {'Mean CV R²':>10} {'Std':>8}")
print('-' * 42)
for name, pipe in models_a.items():
    scores = cross_val_score(pipe, X_a, y_a, cv=cv, scoring='r2')
    print(f"{name:<20} {scores.mean():>10.4f} {scores.std():>8.4f}")

What signal does this hierarchy show?

***

##### Answer

*TLDR: KNN is broken (R² < Dummy) while OLS and Ridge are fine - a model-specific preprocessing failure.*

KNN performs well below DummyRegressor while OLS and Ridge are both near 0.97. This pattern - one model class broken while others that use the same features are unaffected - points to a preprocessing requirement specific to that model. KNN requires scaled features; OLS and Ridge include `StandardScaler` in their pipelines, but KNN (raw) does not.

Before confirming, examine what KNN is actually predicting.

***

In [ ]:
X_train_a, X_test_a, y_train_a, y_test_a = train_test_split(
    X_a, y_a, test_size=0.2, random_state=42
)

knn_raw = KNeighborsRegressor(n_neighbors=5)
knn_raw.fit(X_train_a, y_train_a)
preds_a = knn_raw.predict(X_test_a)

print(f"Prediction range: [{preds_a.min():.3f}, {preds_a.max():.3f}]  std = {preds_a.std():.3f}")
print(f"Actual range:     [{y_test_a.min():.3f}, {y_test_a.max():.3f}]  std = {y_test_a.std():.3f}")

What does the prediction comparison reveal?

***

##### Answer

*TLDR: Prediction spread is compressed to about half the actual spread - income is dominating the distance calculation.*

Prediction std (≈1.1) is roughly half of actual std (≈2.3). This is the compressed prediction variance signal. With income spanning 130,000 units vs age spanning 62 and education spanning 14, income completely dominates the Euclidean distance calculation. KNN finds the 5 neighbors with the most similar income, regardless of age or education. Since y depends on age and education - which are orthogonal to income - each neighbor's y value is essentially random. Averaging 5 random y values reduces variance by 1/√5, but the predictions remain anchored near the income-bracket mean rather than tracking the true signal.

*Planted problem*: the feature with the largest numeric range dominates unscaled distances, making all other features invisible to KNN.

The fix is a Pipeline with `StandardScaler`. After scaling, all four features contribute equally to distance.

***

In [ ]:
models_a_fixed = {
    'KNN (raw)':    make_pipeline(KNeighborsRegressor(n_neighbors=5)),
    'KNN (scaled)': make_pipeline(StandardScaler(), KNeighborsRegressor(n_neighbors=5)),
    'Ridge (CV)':   make_pipeline(StandardScaler(), RidgeCV(alphas=np.logspace(-3, 3, 50))),
}

print(f"{'Model':<20} {'Mean CV R²':>10} {'Std':>8}")
print('-' * 42)
for name, pipe in models_a_fixed.items():
    scores = cross_val_score(pipe, X_a, y_a, cv=cv, scoring='r2')
    print(f"{name:<20} {scores.mean():>10.4f} {scores.std():>8.4f}")

Scaling restores KNN to competitive performance. This is why KNN always goes inside a Pipeline with `StandardScaler` - not as a best practice but as a correctness requirement.

## Scenario B - Non-linear Signal

A dataset generated by the Friedman #1 benchmark: $y = 10 \sin(\pi x_1 x_2) + 20(x_3 - 0.5)^2 + 10 x_4 + 5 x_5 + \varepsilon$. Friedman #1 is a standard synthetic regression benchmark designed to test non-linear methods: the sin term is a multiplicative interaction that no linear model can express, the squared term is a quadratic, and only x₄ and x₅ have a linear relationship with y. All five features are informative - there are no noise columns - which keeps the problem tractable for distance-based methods.

In [ ]:
X_c, y_c = make_friedman1(n_samples=600, n_features=5, noise=0.5, random_state=42)

In [ ]:
# find best KNN configuration before running the full hierarchy
param_grid = {
    'kneighborsregressor__n_neighbors': [3, 5, 7, 10, 15, 20, 30],
    'kneighborsregressor__weights': ['uniform', 'distance'],
}
pipe_knn_c = make_pipeline(StandardScaler(), KNeighborsRegressor())
gs_c = GridSearchCV(pipe_knn_c, param_grid, cv=cv, scoring='r2')
gs_c.fit(X_c, y_c)

print(f"Best KNN: {gs_c.best_params_}  →  CV R² = {gs_c.best_score_:.4f}")

In [ ]:
models_c = {
    'Dummy':      make_pipeline(DummyRegressor(strategy='mean')),
    'OLS':        make_pipeline(StandardScaler(), LinearRegression()),
    'Ridge (CV)': make_pipeline(StandardScaler(), RidgeCV(alphas=np.logspace(-3, 3, 50))),
    'Lasso (CV)': make_pipeline(StandardScaler(), LassoCV(cv=5, random_state=42)),
    'KNN (best)': gs_c.best_estimator_,
}

print(f"{'Model':<20} {'Mean CV R²':>10} {'Std':>8}")
print('-' * 42)
for name, pipe in models_c.items():
    scores = cross_val_score(pipe, X_c, y_c, cv=cv, scoring='r2')
    print(f"{name:<20} {scores.mean():>10.4f} {scores.std():>8.4f}")

What does this hierarchy tell you about the data?

***

##### Answer

*TLDR: KNN substantially outperforms linear models - the data has non-linear structure that linear models cannot express.*

OLS, Ridge, and Lasso all converge to the same R² (the linear components of the Friedman function, captured well by any linear model). KNN with optimal hyperparameters is substantially higher. The gap between the best linear model and KNN is the non-linearity signal: the model class is the bottleneck, not collinearity or data quantity. Linear models have exhausted what they can learn from this data.

The learning curve makes the diagnosis concrete: compare the OLS shape against the best KNN.

***

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

plot_learning_curve(
    make_pipeline(StandardScaler(), LinearRegression()),
    X_c, y_c,
    'OLS - high bias on Friedman data',
    cv, ax=axes[0],
)
plot_learning_curve(
    gs_c.best_estimator_,
    X_c, y_c,
    'KNN (best) - data-limited',
    cv, ax=axes[1],
)
plt.tight_layout()
plt.show()

What do these learning curves confirm about the diagnosis?

***

##### Answer

*TLDR: OLS hits a ceiling (high bias); KNN keeps improving with more data (data-limited). The non-linearity is confirmed.*

OLS shows the high-bias pattern: training and CV scores converge at the same ceiling and stay there regardless of sample size. The model has learned everything it can - adding more data will not move the needle. The ceiling is set by what a linear function can represent.

KNN shows the data-limited pattern: there is a gap between training and CV scores, but it narrows as $n$ grows. More observations would continue to improve performance because larger $n$ allows KNN to find genuinely local neighbors.

*Planted problem*: the true function contains a multiplicative interaction ($\sin(\pi x_1 x_2)$) and a quadratic term ($(x_3 - 0.5)^2$) alongside two linear terms. OLS captures the linear terms (x₄, x₅) but cannot reach the non-linear components, so its ceiling is below the theoretical maximum. Fix: polynomial features or interaction terms (Unit 2 tools), or a non-parametric model family (Decision Trees, Random Forests - Unit 4).

***

## Scenario C - Noise Features

A dataset with 3 strong predictors and a known linear relationship: $y = 2x_1 + x_2 - 1.5x_3 + \varepsilon$. We add irrelevant noise features in increasing quantities and track how each model responds.

In [ ]:
rng_d = np.random.default_rng(4)
n_d = 300

X_d_core = rng_d.standard_normal((n_d, 3))
y_d = 2 * X_d_core[:, 0] + X_d_core[:, 1] - 1.5 * X_d_core[:, 2] + rng_d.normal(0, 0.5, n_d)

# generate the full noise pool upfront; slice different amounts below
X_d_noise = rng_d.standard_normal((n_d, 100))

In [ ]:
noise_counts = [0, 2, 5, 10, 20, 50, 100]
ridge_r2 = []
knn_r2   = []

for k in noise_counts:
    X_d = np.hstack([X_d_core, X_d_noise[:, :k]]) if k > 0 else X_d_core

    pipe_r = make_pipeline(StandardScaler(), RidgeCV(alphas=np.logspace(-3, 3, 50)))
    pipe_k = make_pipeline(StandardScaler(), KNeighborsRegressor(n_neighbors=5))

    ridge_r2.append(cross_val_score(pipe_r, X_d, y_d, cv=cv, scoring='r2').mean())
    knn_r2.append(cross_val_score(pipe_k, X_d, y_d,   cv=cv, scoring='r2').mean())

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))

ax.plot(noise_counts, ridge_r2, 'o-', label='Ridge (CV)')
ax.plot(noise_counts, knn_r2,   's--', label='KNN (k=5)')

ax.set_xlabel('Number of noise features added')
ax.set_ylabel('Mean CV R²')
ax.set_title('Performance vs Noise Features: Ridge vs KNN')
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

Why do Ridge and KNN degrade at such different rates as noise features increase?

***

##### Answer

*TLDR: Ridge shrinks noise coefficients toward zero; KNN's distances become meaningless in high dimensions.*

Ridge degrades gradually because the L2 penalty shrinks noise-feature coefficients toward zero. The model effectively down-weights features that explain little variance. At 100 noise features, regularization is still recovering most of the signal.

KNN degrades steeply because of the curse of dimensionality. Distance is computed across all features simultaneously. Each noise dimension adds variance to the distance calculation without adding predictive information. In high-dimensional space, all points become approximately equidistant - the k nearest neighbors are no longer semantically "close" in any useful sense. The k-neighbor average becomes an average of nearly random y values, approaching DummyRegressor performance.

*Planted problem*: only 3 of the 3-to-103 features carry signal. Fix: perform feature selection before applying KNN - use correlation with the target, feature importance from a linear model, or `SelectKBest` to identify and keep only the informative features. For Ridge, regularization handles this automatically.

***

## Scenario D - Collinear Predictors

This is the most subtle failure mode: the hierarchy reveals nothing unusual. A dataset with four continuous predictors labeled x1 through x4. Run the full hierarchy first.

In [ ]:
rng_b = np.random.default_rng(2)
n_b = 200

x1 = rng_b.normal(0, 1, n_b)
x2 = x1 + rng_b.normal(0, 0.05, n_b)   # nearly identical to x1
x3 = rng_b.normal(0, 1, n_b)
x4 = rng_b.normal(0, 1, n_b)

X_b = pd.DataFrame({'x1': x1, 'x2': x2, 'x3': x3, 'x4': x4})

# ground truth: x1 and x2 together contribute 2.0, x3 is +1.5, x4 is -0.5
y_b = (x1 + x2) + 1.5 * x3 - 0.5 * x4 + rng_b.normal(0, 0.3, n_b)

In [ ]:
models_b = {
    'Dummy':      make_pipeline(DummyRegressor(strategy='mean')),
    'OLS':        make_pipeline(StandardScaler(), LinearRegression()),
    'Ridge (CV)': make_pipeline(StandardScaler(), RidgeCV(alphas=np.logspace(-3, 3, 50))),
    'Lasso (CV)': make_pipeline(StandardScaler(), LassoCV(cv=5, random_state=42)),
}

print(f"{'Model':<20} {'Mean CV R²':>10} {'Std':>8}")
print('-' * 42)
for name, pipe in models_b.items():
    scores = cross_val_score(pipe, X_b, y_b, cv=cv, scoring='r2')
    print(f"{name:<20} {scores.mean():>10.4f} {scores.std():>8.4f}")

What does this hierarchy tell you? What is notable about these results?

***

##### Answer

*TLDR: Nothing is wrong with the hierarchy - that is exactly the point. Collinearity is invisible to predictive metrics.*

OLS, Ridge, and Lasso return identical CV R² to four decimal places. The hierarchy gives no signal that anything is wrong. Collinearity does not hurt predictive accuracy - the combined effect of x1 and x2 is still captured correctly by any of these models. The problem only surfaces when you need to interpret or rely on individual coefficients.

***

In [ ]:
pipe_ols   = make_pipeline(StandardScaler(), LinearRegression())
pipe_ridge = make_pipeline(StandardScaler(), RidgeCV(alphas=np.logspace(-3, 3, 50)))

pipe_ols.fit(X_b, y_b)
pipe_ridge.fit(X_b, y_b)

coef_df = pd.DataFrame({
    'OLS':   pipe_ols.named_steps['linearregression'].coef_.round(3),
    'Ridge': pipe_ridge.named_steps['ridgecv'].coef_.round(3),
}, index=X_b.columns)

print("Coefficients from a single fit (full dataset):")
print(coef_df)
print(f"\nTrue combined effect of x1 + x2: 2.0  →  each should be ~1.0 if split evenly")

The single fit looks fine - both models land near the correct values. But a single fit is misleading when predictors are nearly identical. OLS has to split the combined x1+x2 effect arbitrarily between two indistinguishable features. On this particular dataset it happened to split evenly, but refit on a different sample and the split changes completely. The real problem is coefficient *instability*, not the point estimate.

Refit both models 200 times on bootstrap resamples to measure that instability directly.

In [ ]:
rng_boot = np.random.default_rng(99)
n_boot = 200

ols_boot   = np.zeros((n_boot, 4))
ridge_boot = np.zeros((n_boot, 4))

for i in range(n_boot):
    idx = rng_boot.integers(0, n_b, n_b)   # resample with replacement
    pipe_ols.fit(X_b.iloc[idx].values, y_b[idx])
    pipe_ridge.fit(X_b.iloc[idx].values, y_b[idx])
    ols_boot[i]   = pipe_ols.named_steps['linearregression'].coef_
    ridge_boot[i] = pipe_ridge.named_steps['ridgecv'].coef_

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 4))

for ax, data, title in zip(
    axes,
    [ols_boot, ridge_boot],
    ['OLS Coefficients', 'Ridge Coefficients'],
):
    ax.boxplot(data, tick_labels=X_b.columns, patch_artist=True,
               medianprops={'color': 'black', 'linewidth': 2})
    ax.axhline(0, color='k', linewidth=0.5, linestyle='--')
    ax.set_title(f'{title} ({n_boot} bootstrap samples)')
    ax.set_ylabel('Coefficient (standardized)')
    ax.grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.show()

In [ ]:
print("Coefficient std across bootstrap samples:")
print(pd.DataFrame({
    'OLS std':   ols_boot.std(axis=0).round(3),
    'Ridge std': ridge_boot.std(axis=0).round(3),
}, index=X_b.columns))

What does the bootstrap distribution reveal about OLS vs Ridge on this dataset?

***

##### Answer

*TLDR: OLS x1 and x2 are 5-10x more variable than x3 and x4; Ridge reduces that instability by ~40%. The problem is not the point estimate - it's the variance.*

OLS coefficients for x1 and x2 show 5-10x higher variability across bootstrap samples than the independent features x3 and x4. The boxes in the OLS plot for x1 and x2 are wide; for x3 and x4 they are narrow. Ridge reduces the instability for x1 and x2 by about 40%, while leaving x3 and x4 essentially unchanged. The L2 penalty prefers the equal-split solution, making each sample's allocation less arbitrary.

Neither model completely eliminates the instability when near-perfectly collinear features are present.

***

In [ ]:
print(f"Correlation between x1 and x2: {X_b['x1'].corr(X_b['x2']):.4f}")
print(f"Correlation between x1 and x3: {X_b['x1'].corr(X_b['x3']):.4f}")
print(f"Correlation between x1 and x4: {X_b['x1'].corr(X_b['x4']):.4f}")

*Planted problem*: x2 = x1 + tiny noise, correlation near 1.0. OLS has to allocate the combined x1+x2 effect across two nearly indistinguishable features. The allocation is numerically arbitrary and changes with the sample. This is the "OLS breaks, Ridge shares" pattern from the regularization lecture. The single-fit coefficients may look reasonable on one dataset but will be unreliable on another.

### Fixing Collinearity

Ridge reduces instability but does not eliminate it. For near-perfect collinearity (r > 0.99), the most effective fix is to drop the redundant feature entirely. Best practice:

- r > 0.99: drop one feature (the collinearity is so severe that keeping both adds no information)
- r ≈ 0.7-0.99: use Ridge (regularizes without information loss; uncertainty about which is slightly more informative)
- Let Lasso decide: Lasso's L1 penalty will zero one feature automatically during its path

Dropping x2 means OLS no longer has to split the combined effect. The x1 coefficient absorbs the full contribution and changes from ~1.0 to ~2.0 - this is correct: x1 now represents both its own effect and x2's.

In [ ]:
X_b_fixed = X_b.drop(columns=['x2'])

pipe_ols_fixed = make_pipeline(StandardScaler(), LinearRegression())
pipe_ols_fixed.fit(X_b_fixed, y_b)

print("OLS coefficients after dropping x2:")
print(pd.DataFrame(
    {'coef': pipe_ols_fixed.named_steps['linearregression'].coef_.round(3)},
    index=X_b_fixed.columns,
))
print(f"\nNote: x1 absorbs the combined x1+x2 effect, so its coefficient doubles to ~2.0")

In [ ]:
# bootstrap on the reduced dataset to confirm stability is restored
ols_boot_fixed = np.zeros((n_boot, 3))
for i in range(n_boot):
    idx = rng_boot.integers(0, n_b, n_b)
    pipe_ols_fixed.fit(X_b_fixed.iloc[idx].values, y_b[idx])
    ols_boot_fixed[i] = pipe_ols_fixed.named_steps['linearregression'].coef_

print("Coefficient std after dropping x2 (compare to OLS std above):")
print(pd.DataFrame(
    {'OLS std (x2 dropped)': ols_boot_fixed.std(axis=0).round(3)},
    index=X_b_fixed.columns,
))

In [ ]:
scores_full  = cross_val_score(make_pipeline(StandardScaler(), LinearRegression()), X_b,       y_b, cv=cv, scoring='r2')
scores_fixed = cross_val_score(make_pipeline(StandardScaler(), LinearRegression()), X_b_fixed, y_b, cv=cv, scoring='r2')

print(f"OLS CV R² (x1, x2, x3, x4):  {scores_full.mean():.4f}")
print(f"OLS CV R² (x1, x3, x4 only): {scores_fixed.mean():.4f}")

Dropping x2 restores OLS to the same stability as x3 and x4 with no loss in predictive accuracy - predictive accuracy was never affected, only coefficient reliability.

## Summary

Four failure modes, four signals, four fixes:

- *Scaling failure*: KNN R² near or below DummyRegressor with compressed prediction spread, while OLS/Ridge perform normally. Fix: wrap KNN in `Pipeline(StandardScaler(), ...)` - this is a correctness requirement, not a style choice.
- *Non-linearity*: KNN outperforms the best linear model in the hierarchy; OLS learning curve converges at a low R² ceiling that does not rise with more data. Fix: polynomial features or interaction terms from Unit 2, or non-parametric model families from Unit 4.
- *Curse of dimensionality*: KNN degrades rapidly as irrelevant features are added while Ridge degrades slowly. Fix: feature selection before applying KNN, or prefer linear models when the feature space is large relative to the signal.
- *Collinearity*: hierarchy R² is identical across OLS, Ridge, and Lasso - the signal is absent from predictive metrics; OLS coefficient distributions are wide and unstable across samples while Ridge is tighter. Fix: for r > 0.99, drop one feature; for moderate collinearity, use Ridge; Lasso handles it automatically by zeroing one feature during its regularization path.

`learning_curve` completes the diagnostic toolkit: `validation_curve` asks "is this model too simple or too complex?", `learning_curve` asks "is performance limited by data quantity or model class?" Together with the baseline hierarchy, they cover the most common regression failure modes - which are also the most common project failure modes.